## Imports and Loadings

In [ ]:
import sys
import os

# Get the absolute path of the project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
import yfinance as yf
import logging
import os
import warnings

# --- Disable Streamlit internal logs before it initializes ---
os.environ["STREAMLIT_SUPPRESS_LOGS"] = "1"
os.environ["STREAMLIT_LOGGING_LEVEL"] = "ERROR"

# Configure Streamlit logging more comprehensively
logging.getLogger(
    "streamlit.runtime.caching.cache_data_api").setLevel(logging.ERROR)
logging.getLogger("streamlit").setLevel(logging.ERROR)
logging.getLogger("streamlit.runtime.connection").setLevel(logging.ERROR)
logging.getLogger("streamlit.runtime").setLevel(logging.ERROR)
logging.getLogger("streamlit.elements").setLevel(logging.ERROR)

# Disable all existing loggers
logging.disable(logging.WARNING)

# --- Suppress Python warnings ---
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="No runtime found")
warnings.filterwarnings("ignore", message=".*auto_adjust.*")
warnings.filterwarnings("ignore", message="YF.download()")

# Also suppress the download progress output
# Suppress the progress bar by monkey-patching tqdm if needed
try:
    from tqdm import tqdm
    tqdm._instances.clear()
except:
    pass

In [ ]:

import numpy as np
import pandas as pd
import streamlit as st
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import MinMaxScaler, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from data.data_loader import load_data

In [ ]:
# df = load_data('BTC-USD')
from pandas import read_csv

In [ ]:
print(os.getcwd())

In [ ]:
df = read_csv('../data/BTC-USD/BTC-USD_from_2010-01-01_till_2026-01-03.csv')
df.head()

## Understand The Dataset

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
# Set Date as index for time series analysis

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Data Cleaning

### Missing Values and Duplicates

In [ ]:
print("before handling missing values:")
print(df.isna().sum()) # Check for missing values

df.ffill(inplace=True)  # Filling the missing values with Forward fill method

print("after handling missing values:")
print(df.isna().sum())  # Check for missing values

In [ ]:
print("before handling duplicate values:")
print(df.isna().sum())  # Check for duplicate values

df.drop_duplicates(inplace=True)  # Remove duplicate rows if any

print("after handling duplicate values:")
print(df.isna().sum())  # Check for missing values

### Removing The Outliers

In [ ]:
lower_limit = df['Close'].quantile(0.05)
upper_limit = df['Close'].quantile(0.95)

print(f"Lower Limit {lower_limit} and Upper Limit {upper_limit}")

#### Trimming

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df_trimmed = df[
                (lower_limit <= df['Close']) & 
                (df['Close'] <= upper_limit)
              ]
df_trimmed.shape

#### Capping - Winsorization Method

winsorization doesn't remove any rows! It only replaces extreme values with the percentile limits.

In [ ]:
df.shape

In [ ]:
df.tail()

In [ ]:
df_winsorized = df.copy()

In [ ]:
df_winsorized['Close'] = df['Close'].clip(lower=lower_limit, upper=upper_limit)

In [ ]:
df_winsorized.tail()

In [ ]:
df_winsorized.shape

In [ ]:
df.tail()

In [ ]:
df_winsorized.head()

In [ ]:
df['Close'].tail()

In [ ]:
df_winsorized['Close'].tail()

## Univariate Analysis

In [ ]:
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

# Create figure with 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Histogram with KDE
sns.histplot(df['Close'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of Close Prices')
axes[0].set_xlabel('Close Price')
axes[0].set_ylabel('Frequency')

# Plot 2: Box plot
sns.boxplot(y=df['Close'], ax=axes[1])
axes[1].set_title('Box Plot of Close Prices')
axes[1].set_ylabel('Close Price')

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

# Create figure with 3 rows, 2 columns
fig, axes = plt.subplots(3, 2, figsize=(15, 15))

# ===== ORIGINAL DATA =====
# Histogram - Original
sns.histplot(df['Close'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Original Data: Distribution of Close Prices')
axes[0, 0].set_xlabel('Close Price')
axes[0, 0].set_ylabel('Frequency')

# Box plot - Original
sns.boxplot(y=df['Close'], ax=axes[0, 1])
axes[0, 1].set_title('Original Data: Box Plot of Close Prices')
axes[0, 1].set_ylabel('Close Price')

# ===== TRIMMED DATA =====
# Histogram - Trimmed
sns.histplot(df_trimmed['Close'], kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Trimmed Data: Distribution of Close Prices')
axes[1, 0].set_xlabel('Close Price')
axes[1, 0].set_ylabel('Frequency')

# Box plot - Trimmed
sns.boxplot(y=df_trimmed['Close'], ax=axes[1, 1])
axes[1, 1].set_title('Trimmed Data: Box Plot of Close Prices')
axes[1, 1].set_ylabel('Close Price')

# ===== WINSORIZED DATA =====
# Histogram - Winsorized
sns.histplot(df_winsorized['Close'], kde=True, ax=axes[2, 0])
axes[2, 0].set_title('Winsorized Data: Distribution of Close Prices')
axes[2, 0].set_xlabel('Close Price')
axes[2, 0].set_ylabel('Frequency')

# Box plot - Winsorized
sns.boxplot(y=df_winsorized['Close'], ax=axes[2, 1])
axes[2, 1].set_title('Winsorized Data: Box Plot of Close Prices')
axes[2, 1].set_ylabel('Close Price')

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
df_winsorized.tail()

## Checking the normality

In [ ]:
from scipy.stats import kurtosis, skew

print("=" * 75)
print("KURTOSIS & SKEWNESS COMPARISON")
print("=" * 75)

print("\nDataset       Shape       Rows    Removed   Kurtosis   Skewness")
print("-" * 75)

print(
    f"{'Original':<12} {str(df.shape):<10} {len(df):<8} {'0':<9} {kurtosis(df['Close']):>9.4f} {skew(df['Close']):>10.4f}")

rows_removed = len(df) - len(df_trimmed)
print(
    f"{'Trimmed':<12} {str(df_trimmed.shape):<10} {len(df_trimmed):<8} {rows_removed:<9} {kurtosis(df_trimmed['Close']):>9.4f} {skew(df_trimmed['Close']):>10.4f}")

print(
    f"{'Winsorized':<12} {str(df_winsorized.shape):<10} {len(df_winsorized):<8} {'0':<9} {kurtosis(df_winsorized['Close']):>9.4f} {skew(df_winsorized['Close']):>10.4f}")

print("-" * 75)

# Add percentage change
orig_kurt = kurtosis(df['Close'])
orig_skew = skew(df['Close'])

print(f"\nPercentage Reduction from Original:")
print(
    f"  Trimmed:    Kurtosis: {((orig_kurt - kurtosis(df_trimmed['Close'])) / orig_kurt * 100):.1f}% | Skewness: {((orig_skew - skew(df_trimmed['Close'])) / orig_skew * 100):.1f}%")
print(
    f"  Winsorized: Kurtosis: {((orig_kurt - kurtosis(df_winsorized['Close'])) / orig_kurt * 100):.1f}% | Skewness: {((orig_skew - skew(df_winsorized['Close'])) / orig_skew * 100):.1f}%")

### Applying Histogram and QQ Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Create a 3x2 subplot grid (3 rows for datasets, 2 columns for plot types)
fig, axes = plt.subplots(3, 2, figsize=(15, 15))

# List of datasets and their titles
datasets = [
    (df['Close'].dropna(), 'Original Data'),
    (df_trimmed['Close'].dropna(), 'Trimmed Data'),
    (df_winsorized['Close'].dropna(), 'Winsorized Data')
]

# Colors for each dataset
colors = ['skyblue', 'lightcoral', 'lightgreen']

for i, (data, title) in enumerate(datasets):
    color = colors[i]

    # --- Left Column: Histogram with KDE ---
    ax_hist = axes[i, 0]
    sns.histplot(data, kde=True, bins=50, color=color, ax=ax_hist)
    ax_hist.set_title(f'{title}\nHistogram', fontsize=14, fontweight='bold')
    ax_hist.set_xlabel('Close Price')
    ax_hist.set_ylabel('Frequency')

    # Add mean and median lines
    mean_val = data.mean()
    median_val = data.median()
    ax_hist.axvline(mean_val, color='red', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_val:.2f}')
    ax_hist.axvline(median_val, color='green', linestyle='--', linewidth=1.5,
                    label=f'Median: {median_val:.2f}')
    ax_hist.legend()

    # Calculate and display skewness and kurtosis
    skew_val = stats.skew(data)
    kurt_val = stats.kurtosis(data)
    ax_hist.text(0.02, 0.98, f'Skewness: {skew_val:.3f}\nKurtosis: {kurt_val:.3f}',
                 transform=ax_hist.transAxes, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # --- Right Column: Q-Q Plot ---
    ax_qq = axes[i, 1]

    # Use probplot with fit=True to get the line parameters
    qq_data = stats.probplot(data, dist="norm", fit=True)

    # Extract values - qq_data returns ((osm, osr), (slope, intercept, r))
    osm = qq_data[0][0]  # theoretical quantiles
    osr = qq_data[0][1]  # ordered data values
    slope = qq_data[1][0]  # slope of the fit line
    intercept = qq_data[1][1]  # intercept of the fit line
    r = qq_data[1][2]  # correlation coefficient

    # Plot the Q-Q plot manually
    ax_qq.scatter(osm, osr, alpha=0.7, color=color, s=20)

    # Add the fitted line
    ax_qq.plot(osm, slope * osm + intercept, 'r-', alpha=0.8,
               label=f'Fit line (r={r:.3f})')

    # Add reference line for perfect normality
    x_limits = [osm.min(), osm.max()]
    ax_qq.plot(x_limits, x_limits, 'k--', alpha=0.5, label='Perfect Normal')

    ax_qq.set_title(f'{title}\nQ-Q Plot', fontsize=14, fontweight='bold')
    ax_qq.set_xlabel('Theoretical Quantiles')
    ax_qq.set_ylabel('Ordered Values')
    ax_qq.grid(True, alpha=0.3)
    ax_qq.legend()

# Add an overall title
plt.suptitle('Normality Analysis: Original vs Trimmed vs Winsorized Data',
             fontsize=16, fontweight='bold', y=0.98)

# Adjust layout
plt.tight_layout()
plt.show()

# Additional: Combined Q-Q plots for comparison
print("\n" + "="*60)
print("COMBINED Q-Q PLOT COMPARISON")
print("="*60)

plt.figure(figsize=(10, 8))

# Colors for the combined plot
plot_colors = ['blue', 'red', 'green']
plot_labels = ['Original', 'Trimmed', 'Winsorized']
plot_data = [df['Close'].dropna(), df_trimmed['Close'].dropna(),
             df_winsorized['Close'].dropna()]

# Plot all three Q-Q plots together for direct comparison
for data, color, label in zip(plot_data, plot_colors, plot_labels):
    qq_result = stats.probplot(data, dist="norm", fit=True)
    osm, osr = qq_result[0]
    slope, intercept, r = qq_result[1]

    plt.scatter(osm, osr, alpha=0.5, s=20, color=color,
                label=f'{label} (r={r:.3f}, n={len(data)})')
    plt.plot(osm, slope * osm + intercept, color=color, alpha=0.7, linewidth=1)

# Add reference line
plt.plot([-3, 3], [-3, 3], 'k--', alpha=0.5,
         linewidth=2, label='Perfect Normal')

plt.title('Combined Q-Q Plot Comparison: Close Prices',
          fontsize=14, fontweight='bold')
plt.xlabel('Theoretical Quantiles (Normal)')
plt.ylabel('Ordered Values (Close Price)')
plt.grid(True, alpha=0.3)
plt.legend(loc='best')
plt.tight_layout()
plt.show()


### Applying YeoJohnson Transformations

In [ ]:
from sklearn.preprocessing import (
    FunctionTransformer, PowerTransformer,
    StandardScaler, MinMaxScaler, RobustScaler
)

In [ ]:
df_transformed = df.copy()  # Create a copy of the original DataFrame

In [ ]:
df_transformed.columns

In [ ]:
# Transform and replace original columns
for column in df.columns.tolist():
    transformer = PowerTransformer(method='yeo-johnson')
    df_transformed[column] = transformer.fit_transform(df_transformed[[column]])

print("✅ Original columns replaced with Yeo-Johnson transformed values")
print(f"Current columns: {list(df_transformed.columns)}")

In [ ]:
df_transformed.head()

In [ ]:
# transformations = {
#     'Log': FunctionTransformer(np.log1p),
#     'Sqrt': FunctionTransformer(np.sqrt),
#     'Cbrt': FunctionTransformer(np.cbrt),
#     'Reciprocal': FunctionTransformer(lambda x: 1 / (x + 1e-9)),
#     # Yeo-Johnson works with zero/negative values
#     'YeoJohnson': PowerTransformer(method='yeo-johnson'),
#     # Box-Cox requires all positive data
#     'BoxCox': PowerTransformer(method='box-cox'),
#     'Standard': StandardScaler(),
#     'MinMax': MinMaxScaler(),
#     'Robust': RobustScaler()
# }

### Plotting Yeojohnson close column

In [ ]:
from sklearn.preprocessing import PowerTransformer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Apply Yeo-Johnson transformation to 'Close' column in df
yeojohnson_transformer = PowerTransformer(method='yeo-johnson')

# Fit and transform the 'Close' column from original df
close = yeojohnson_transformer.fit_transform(df[['Close']])

# Add the transformed column to df_transformed dataframe
df_transformed['Close'] = close

print("✅ Yeo-Johnson transformation applied successfully!")
print(f"Original df shape: {df.shape}")
print(f"Transformed df shape: {df_transformed.shape}")
print(f"New column added to df_transformed: 'Close'")

# Calculate skewness and kurtosis before/after transformation
original_skew = stats.skew(df['Close'])
original_kurt = stats.kurtosis(df['Close'])
transformed_skew = stats.skew(df_transformed['Close'])
transformed_kurt = stats.kurtosis(df_transformed['Close'])

print("\n" + "="*60)
print("BEFORE/AFTER TRANSFORMATION COMPARISON")
print("="*60)
print(f"{'Metric':<15} {'Original (df)':<15} {'Yeo-Johnson (df_transformed)':<15} {'Improvement':<15}")
print("-" * 65)
print(f"{'Skewness':<15} {original_skew:<15.4f} {transformed_skew:<15.4f} {abs(original_skew) - abs(transformed_skew):<15.4f}")
print(f"{'Kurtosis':<15} {original_kurt:<15.4f} {transformed_kurt:<15.4f} {abs(original_kurt) - abs(transformed_kurt):<15.4f}")

# Visualize the transformation
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Original Data (from df)
# Histogram
axes[0, 0].hist(df['Close'], bins=50, color='skyblue',
                alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Original Close Price\nHistogram (df)', fontweight='bold')
axes[0, 0].set_xlabel('Price')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].text(0.02, 0.98, f'Skew: {original_skew:.3f}\nKurt: {original_kurt:.3f}',
                transform=axes[0, 0].transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Q-Q Plot for Original
qq_orig = stats.probplot(df['Close'], dist="norm", fit=True)
axes[0, 1].scatter(qq_orig[0][0], qq_orig[0][1],
                   alpha=0.6, s=20, color='skyblue')
axes[0, 1].plot(qq_orig[0][0], qq_orig[1][0] * qq_orig[0][0] + qq_orig[1][1],
                'r-', alpha=0.8, label=f'Fit (r={qq_orig[1][2]:.3f})')
axes[0, 1].plot([-3, 3], [-3, 3], 'k--', alpha=0.5, label='Perfect Normal')
axes[0, 1].set_title('Original Close Price\nQ-Q Plot (df)', fontweight='bold')
axes[0, 1].set_xlabel('Theoretical Quantiles')
axes[0, 1].set_ylabel('Ordered Values')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Box plot
axes[0, 2].boxplot(df['Close'].dropna(), vert=True, patch_artist=True,
                   boxprops=dict(facecolor='skyblue'))
axes[0, 2].set_title('Original Close Price\nBox Plot (df)', fontweight='bold')
axes[0, 2].set_ylabel('Price')

# Row 2: Yeo-Johnson Transformed Data (from df_transformed)
# Histogram
axes[1, 0].hist(df_transformed['Close'], bins=50,
                color='lightgreen', alpha=0.7, edgecolor='black')
axes[1, 0].set_title(
    'Yeo-Johnson Transformed\nHistogram (df_transformed)', fontweight='bold')
axes[1, 0].set_xlabel('Transformed Price')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].text(0.02, 0.98, f'Skew: {transformed_skew:.3f}\nKurt: {transformed_kurt:.3f}',
                transform=axes[1, 0].transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Q-Q Plot for Transformed
qq_trans = stats.probplot(
    df_transformed['Close'], dist="norm", fit=True)
axes[1, 1].scatter(qq_trans[0][0], qq_trans[0][1],
                   alpha=0.6, s=20, color='lightgreen')
axes[1, 1].plot(qq_trans[0][0], qq_trans[1][0] * qq_trans[0][0] + qq_trans[1][1],
                'r-', alpha=0.8, label=f'Fit (r={qq_trans[1][2]:.3f})')
axes[1, 1].plot([-3, 3], [-3, 3], 'k--', alpha=0.5, label='Perfect Normal')
axes[1, 1].set_title(
    'Yeo-Johnson Transformed\nQ-Q Plot (df_transformed)', fontweight='bold')
axes[1, 1].set_xlabel('Theoretical Quantiles')
axes[1, 1].set_ylabel('Ordered Values')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Box plot
axes[1, 2].boxplot(df_transformed['Close'].dropna(), vert=True, patch_artist=True,
                   boxprops=dict(facecolor='lightgreen'))
axes[1, 2].set_title(
    'Yeo-Johnson Transformed\nBox Plot (df_transformed)', fontweight='bold')
axes[1, 2].set_ylabel('Transformed Price')

plt.suptitle('Yeo-Johnson Transformation: Before (df) vs After (df_transformed)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Additional: Show correlation between original and transformed
print("\n" + "="*60)
print("CORRELATION: Original vs Transformed")
print("="*60)

# Create a temporary dataframe with both values
temp_df = pd.DataFrame({
    'Original_Close': df['Close'],
    'Transformed_Close': df_transformed['Close']
})

correlation = temp_df.corr().iloc[0, 1]
print(
    f"Correlation between original Close and Yeo-Johnson transformed: {correlation:.4f}")

if correlation < 0:
    print("⚠️ Note: Negative correlation indicates the transformation inverted the values")
    print(f"Lambda parameter: {yeojohnson_transformer.lambdas_[0]:.4f}")

In [ ]:
df_transformed.head()

### Previous

In [ ]:
df_transformed.head()

In [ ]:
# # Loop over all columns that end with '_YeoJohnson'
# for col in [c for c in df_new.columns if c.endswith('_YeoJohnson')]:
#     # Create subplots
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

#     # --- Left: Histogram with KDE ---
#     sns.histplot(df_new[col].dropna(), kde=True, bins=50, ax=ax1)
#     ax1.set_title(f'Distribution of {col} Column')
#     ax1.set_xlabel(col)
#     ax1.set_ylabel('Frequency')

#     # --- Right: Q–Q Plot ---
#     stats.probplot(df_new[col].dropna(), dist="norm", plot=ax2)
#     ax2.set_title(f"Q–Q Plot of {col} Column")
#     ax2.set_xlabel('Theoretical Quantiles')
#     ax2.set_ylabel('Ordered Values')

#     # Adjust layout and display
#     plt.tight_layout()
#     plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# # Loop through all columns ending with '_YeoJohnson' to create boxplots
# for col in [c for c in df_new.columns if c.endswith('_YeoJohnson')]:
#     plt.figure()
#     sns.boxplot(x=df_new[col], orient='h')
#     plt.title(f'Boxplot of {col}')
#     plt.xlabel(col)
#     plt.tight_layout()
#     plt.show()

In [ ]:
# # Loop through all columns ending with '_YeoJohnson'  to create histograms
# for col in [c for c in df_new.columns if c.endswith('_YeoJohnson')]:
#     plt.figure()
#     plt.hist(df_new[col], bins=30, edgecolor='black')
#     plt.title(f'Histogram of {col}')
#     plt.xlabel(col)
#     plt.ylabel('Frequency')
#     plt.tight_layout()
#     plt.show()

#### Trimming

In [ ]:
# df_new.columns

In [ ]:
# upper_limit = df_new['Volume_YeoJohnson'].quantile(0.95)
# upper_limit

In [ ]:
# lower_limit = df_new['Volume_YeoJohnson'].quantile(0.05)
# lower_limit

In [ ]:
# df_updated = df_new[(lower_limit <= df_new['Volume_YeoJohnson']) &
#        (df_new['Volume_YeoJohnson'] <= upper_limit)]

In [ ]:
# df_updated.shape

In [ ]:
# df_new.shape

In [ ]:
# df_updated.columns

In [ ]:
# # Loop through all df_updated columns ending with '_YeoJohnson' to create boxplots
# for col in [c for c in df_updated.columns if c.endswith('_YeoJohnson')]:
#     plt.figure()
#     sns.boxplot(x=df_updated[col], orient='h')
#     plt.title(f'Boxplot of {col}')
#     plt.xlabel(col)
#     plt.tight_layout()
#     plt.show()

In [ ]:
# # Loop through all columns ending with '_YeoJohnson'
# for col in [c for c in df_updated.columns if c.endswith('_YeoJohnson')]:
#     plt.figure()
#     plt.hist(df_updated[col], bins=30, edgecolor='black')
#     plt.title(f'Histogram of {col}')
#     plt.xlabel(col)
#     plt.ylabel('Frequency')
#     plt.tight_layout()
#     plt.show()

In [ ]:
# df_updated.shape

#### Capping - Winsorization Method

In [ ]:

# # Loop through all columns ending with '_YeoJohnson'
# for col in [c for c in df_new.columns if c.endswith('_YeoJohnson')]:
#     fig, axes = plt.subplots(1, 2, figsize=(10, 4))

#     # Histogram (left side)
#     sns.histplot(df_new[col], bins=30, kde=True, ax=axes[0])
#     axes[0].set_title(f'Histogram of {col}')
#     axes[0].set_xlabel(col)
#     axes[0].set_ylabel('Frequency')

#     # Boxplot (right side)
#     sns.boxplot(x=df_new[col], orient='h', ax=axes[1])
#     axes[1].set_title(f'Boxplot of {col}')
#     axes[1].set_xlabel(col)

#     plt.tight_layout()
#     plt.show()

In [ ]:
# lower_limit = df_new['Volume_YeoJohnson'].quantile(0.05)
# upper_limit = df_new['Volume_YeoJohnson'].quantile(0.95)

In [ ]:
# df_winsorized = df_new.copy()

In [ ]:
# df_winsorized['Volume_YeoJohnson'] = np.where(df_new['Volume_YeoJohnson'] <= lower_limit, 
#       lower_limit, 
#       np.where(df_new['Volume_YeoJohnson'] >= upper_limit,
#       upper_limit,
#       df_new['Volume_YeoJohnson']))

In [ ]:
# # Loop through all columns ending with '_YeoJohnson'
# for col in [c for c in df_winsorized.columns if c.endswith('_YeoJohnson')]:
#     fig, axes = plt.subplots(1, 2, figsize=(10, 4))

#     # Histogram (left side)
#     sns.histplot(df_winsorized[col], bins=30, kde=True, ax=axes[0])
#     axes[0].set_title(f'Histogram of Winsorized {col}')
#     axes[0].set_xlabel(col)
#     axes[0].set_ylabel('Frequency')

#     # Boxplot (right side)
#     sns.boxplot(x=df_winsorized[col], orient='h', ax=axes[1])
#     axes[1].set_title(f'Boxplot of Winsorized {col}')
#     axes[1].set_xlabel(col)

#     plt.tight_layout()
#     plt.show()

## Multivariate Analysis

In [ ]:
df_transformed.head()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_transformed.corr(numeric_only=True), annot=True,
            cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Features')
plt.show()

# --- Correlation Matrix Explaination ---

# OHLC features (Open, High, Low, Close) show almost perfect correlation (~1.00) with each other

# If the market is up, all four go up.
# If the market is down, all four go down.
# When trading volume increases, price tends to be higher.

## Time Series Specific Checks

#### Trend Visualization

In [ ]:
# Create side-by-side comparison figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left Plot: Original Close Price from df ---
ax1 = axes[0]
ax1.plot(df.index if isinstance(df.index, pd.DatetimeIndex) else df['Date'],
         df['Close'], color='blue', linewidth=2, alpha=0.8)
ax1.set_title('Original Close Price\n(df)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Close Price', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True, alpha=0.3)

# Add statistics text box
orig_stats = f"""Statistics:
Mean: ${df['Close'].mean():.2f}
Std: ${df['Close'].std():.2f}
Min: ${df['Close'].min():.2f}
Max: ${df['Close'].max():.2f}
Skew: {stats.skew(df['Close']):.3f}
Kurt: {stats.kurtosis(df['Close']):.3f}"""

ax1.text(0.02, 0.98, orig_stats, transform=ax1.transAxes,
         verticalalignment='top', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# --- Right Plot: Transformed Close Price from df_transformed ---
ax2 = axes[1]
ax2.plot(df_transformed.index if isinstance(df_transformed.index, pd.DatetimeIndex) else df_transformed['Date'],
         df_transformed['Close'], color='green', linewidth=2, alpha=0.8)
ax2.set_title('Transformed Close Price\n(df_transformed)',
              fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Transformed Value', color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax2.grid(True, alpha=0.3)

# Add statistics text box for transformed data
trans_stats = f"""Statistics:
Mean: {df_transformed['Close'].mean():.3f}
Std: {df_transformed['Close'].std():.3f}
Min: {df_transformed['Close'].min():.3f}
Max: {df_transformed['Close'].max():.3f}
Skew: {stats.skew(df_transformed['Close']):.3f}
Kurt: {stats.kurtosis(df_transformed['Close']):.3f}"""

ax2.text(0.02, 0.98, trans_stats, transform=ax2.transAxes,
         verticalalignment='top', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

# Overall title
plt.suptitle('Side-by-Side Comparison: Original vs Transformed Close Prices',
             fontsize=16, fontweight='bold', y=1.02)

# Adjust layout
plt.tight_layout()
plt.show()

# Additional: Overlay comparison
print("\n" + "="*60)
print("OVERLAY COMPARISON")
print("="*60)

plt.figure(figsize=(14, 6))

# Plot both on same axes with dual y-axis
fig, ax1 = plt.subplots(figsize=(14, 6))

# Original data (left y-axis)
color_orig = 'blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Original Close Price', color=color_orig)
line1 = ax1.plot(df.index if isinstance(df.index, pd.DatetimeIndex) else df['Date'],
                 df['Close'], color=color_orig, linewidth=2, alpha=0.7, label='Original')
ax1.tick_params(axis='y', labelcolor=color_orig)

# Create second y-axis for transformed data
ax2 = ax1.twinx()
color_trans = 'green'
ax2.set_ylabel('Transformed Close Price', color=color_trans)
line2 = ax2.plot(df_transformed.index if isinstance(df_transformed.index, pd.DatetimeIndex) else df_transformed['Date'],
                 df_transformed['Close'], color=color_trans, linewidth=2, alpha=0.7, label='Transformed')
ax2.tick_params(axis='y', labelcolor=color_trans)

# Add legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.title('Overlay Comparison: Original vs Transformed Close Prices',
          fontsize=16, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print correlation between original and transformed
print("\n" + "="*60)
print("CORRELATION ANALYSIS")
print("="*60)

correlation = np.corrcoef(df['Close'], df_transformed['Close'])[0, 1]
print(f"Correlation coefficient: {correlation:.4f}")

if correlation > 0.9:
    strength = "VERY STRONG"
elif correlation > 0.7:
    strength = "STRONG"
elif correlation > 0.5:
    strength = "MODERATE"
elif correlation > 0.3:
    strength = "WEAK"
else:
    strength = "VERY WEAK"

direction = "positive" if correlation > 0 else "negative"
print(f"Relationship: {strength} {direction} correlation")

# Scatter plot to visualize relationship
plt.figure(figsize=(10, 6))
plt.scatter(df['Close'], df_transformed['Close'],
            alpha=0.6, s=20, color='purple')
plt.xlabel('Original Close Price')
plt.ylabel('Transformed Close Price')
plt.title(f'Relationship: Original vs Transformed\nCorrelation = {correlation:.4f}',
          fontsize=14, fontweight='bold')

# Add regression line
z = np.polyfit(df['Close'], df_transformed['Close'], 1)
p = np.poly1d(z)
plt.plot(df['Close'], p(df['Close']), "r--", alpha=0.8,
         label=f'Regression line: y = {z[0]:.3f}x + {z[1]:.3f}')

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Stationarity Check

In [ ]:
# Simple KPSS and ADF tests for Close columns only
from statsmodels.tsa.stattools import kpss, adfuller

print("="*50)
print("STATIONARITY TESTS FOR CLOSE COLUMNS")
print("="*50)

# Test both dataframes
for df_name, df_data in [("Original (df)", df), ("Transformed (df_transformed)", df_transformed)]:
    print(f"\n{df_name}:")

    if 'Close' not in df_data.columns:
        print("  'Close' column not found")
        continue

    # KPSS Test with error handling
    try:
        kpss_stat, kpss_p, kpss_lags, _ = kpss(
            df_data['Close'].dropna(), regression='c', nlags="auto")
        if kpss_p < 0.01:  # If p-value is extremely small
            print(
                f"  KPSS: stat={kpss_stat:.4f}, p<0.01, stationary=No (STRONGLY NON-STATIONARY)")
        else:
            print(
                f"  KPSS: stat={kpss_stat:.4f}, p={kpss_p:.4f}, stationary={'Yes' if kpss_p >= 0.05 else 'No'}")
    except Exception as e:
        print(f"  KPSS: Failed - {str(e)}")

    # ADF Test
    try:
        adf_result = adfuller(df_data['Close'].dropna())
        print(
            f"  ADF: stat={adf_result[0]:.4f}, p={adf_result[1]:.4f}, stationary={'Yes' if adf_result[1] < 0.05 else 'No'}")
    except Exception as e:
        print(f"  ADF: Failed - {str(e)}")

print("\n" + "="*50)
print("INTERPRETATION")
print("="*50)
print("KPSS Warning: Extremely high test statistic → STRONG non-stationarity")
print("KPSS: p≥0.05 = Stationary, p<0.05 = Non-stationary")
print("ADF: p<0.05 = Stationary, p≥0.05 = Non-stationary")

In [ ]:
# Apply first-order differencing
df['Close_diff'] = df['Close'].diff().dropna()
df_transformed['Close_diff'] = df_transformed['Close'].diff().dropna()

# Then retest stationarity

In [ ]:
df_transformed.head()

In [ ]:
df_transformed.columns

In [ ]:
# Simple KPSS and ADF tests for Close_diff columns only
from statsmodels.tsa.stattools import kpss, adfuller

print("="*50)
print("STATIONARITY TESTS FOR Close_diff COLUMNS")
print("="*50)

# Test both dataframes
for df_name, df_data in [("Original (df)", df), ("Transformed (df_transformed)", df_transformed)]:
    print(f"\n{df_name}:")

    if 'Close_diff' not in df_data.columns:
        print("  'Close_diff' column not found")
        continue

    # KPSS Test with error handling
    try:
        kpss_stat, kpss_p, kpss_lags, _ = kpss(
            df_data['Close_diff'].dropna(), regression='c', nlags="auto")
        if kpss_p < 0.01:  # If p-value is extremely small
            print(
                f"  KPSS: stat={kpss_stat:.4f}, p<0.01, stationary=No (STRONGLY NON-STATIONARY)")
        else:
            print(
                f"  KPSS: stat={kpss_stat:.4f}, p={kpss_p:.4f}, stationary={'Yes' if kpss_p >= 0.05 else 'No'}")
    except Exception as e:
        print(f"  KPSS: Failed - {str(e)}")

    # ADF Test
    try:
        adf_result = adfuller(df_data['Close_diff'].dropna())
        print(
            f"  ADF: stat={adf_result[0]:.4f}, p={adf_result[1]:.4f}, stationary={'Yes' if adf_result[1] < 0.05 else 'No'}")
    except Exception as e:
        print(f"  ADF: Failed - {str(e)}")

# print("\n" + "="*50)
# print("INTERPRETATION")
# print("="*50)
# print("KPSS Warning: Extremely high test statistic → STRONG non-stationarity")
# print("KPSS: p≥0.05 = Stationary, p<0.05 = Non-stationary")
# print("ADF: p<0.05 = Stationary, p≥0.05 = Non-stationary")

## Creating new features

In [ ]:
df_transformed = df_transformed[['High', 'Low', 'Open', 'Volume', 'Close']]
df_transformed.head()

In [ ]:
import pandas as pd
import numpy as np

# Moving Averages - Trend following indicators
df_transformed['MA_5'] = df_transformed['Close'].rolling(
    window=5).mean()  # 5-day moving average
df_transformed['MA_10'] = df_transformed['Close'].rolling(
    window=10).mean()  # 10-day moving average
df_transformed['MA_20'] = df_transformed['Close'].rolling(
    window=20).mean()  # 20-day moving average (monthly trend)
df_transformed['MA_50'] = df_transformed['Close'].rolling(
    window=50).mean()  # 50-day moving average (long-term trend)

# Create lagged features - Historical price patterns
for lag in [1, 2, 3, 5, 10, 20, 50]:
    df_transformed[f'Close_lag_{lag}'] = df_transformed['Close'].shift(
        lag)  # Previous day's closing price
    df_transformed[f'Volume_lag_{lag}'] = df_transformed['Volume'].shift(
        lag)  # Historical trading volume
    df_transformed[f'High_lag_{lag}'] = df_transformed['High'].shift(
        lag)  # Previous day's high price
    df_transformed[f'Low_lag_{lag}'] = df_transformed['Low'].shift(
        lag)  # Previous day's low price

# Price relationships - Intraday price dynamics
df_transformed['High_Low_Ratio'] = df_transformed['High'] / \
    df_transformed['Low']  # Daily price range ratio
df_transformed['Close_Open_Ratio'] = df_transformed['Close'] / \
    df_transformed['Open']  # Day's price movement strength
df_transformed['Price_Range'] = df_transformed['High'] - \
    df_transformed['Low']  # Absolute daily price range
# Candle body size (opening to closing movement)
df_transformed['Body_Size'] = abs(
    df_transformed['Close'] - df_transformed['Open'])

# Price position within day's range - Where close price falls in daily range
df_transformed['Price_Position'] = (df_transformed['Close'] - df_transformed['Low']) / \
                                   (df_transformed['High'] - df_transformed['Low'] +
                                    # Normalized close position (0=low, 1=high)
                                    1e-10)

# Extract time features from index - Calendar effects
if isinstance(df_transformed.index, pd.DatetimeIndex):
    # Monday=0, Sunday=6 (weekly patterns)
    df_transformed['Day_of_Week'] = df_transformed.index.dayofweek
    # Month of year (1-12)
    df_transformed['Month'] = df_transformed.index.month
    # Financial quarter (1-4)
    df_transformed['Quarter'] = df_transformed.index.quarter
    df_transformed['Year'] = df_transformed.index.year  # Year
    # Week number (1-52)
    df_transformed['Week_of_Year'] = df_transformed.index.isocalendar().week
    # Day number in year (1-365)
    df_transformed['Day_of_Year'] = df_transformed.index.dayofyear

    # Time of month features - Month-end and beginning effects
    df_transformed['Month_End'] = (df_transformed.index.days_in_month -
                                   # Last 5 days of month
                                   df_transformed.index.day <= 5).astype(int)
    df_transformed['Month_Begin'] = (
        df_transformed.index.day <= 5).astype(int)  # First 5 days of month

# Volatility measures - Risk and price fluctuation indicators
# Percentage price change from previous day
df_transformed['Daily_Return'] = df_transformed['Close'].pct_change()
df_transformed['Volatility_5'] = df_transformed['Daily_Return'].rolling(
    5).std()  # 5-day volatility (short-term risk)
df_transformed['Volatility_10'] = df_transformed['Daily_Return'].rolling(
    10).std()  # 10-day volatility
df_transformed['Volatility_20'] = df_transformed['Daily_Return'].rolling(
    20).std()  # 20-day volatility (monthly risk measure)

# Log returns - Alternative return calculation for better statistical properties
# Add a tiny value to avoid zero division and negative values
epsilon = 1e-10
df_transformed['Log_Return'] = np.log(
    (df_transformed['Close'] + epsilon) /
    (df_transformed['Close'].shift(1) + epsilon)
)

# Create target variable - What we want to predict
forecast_horizon = 1  # Predict next day's close
# Tomorrow's closing price
df_transformed['Target'] = df_transformed['Close'].shift(-forecast_horizon)

# Alternatively: Predict price change - Direction and magnitude
df_transformed['Target_Change'] = df_transformed['Close'].shift(
    # Absolute price change tomorrow
    -forecast_horizon) - df_transformed['Close']

# Binary target for classification (up/down) - Simplified prediction task
df_transformed['Target_Binary'] = (df_transformed['Target_Change'] > 0).astype(
    int)  # 1 if price goes up, 0 if down

In [ ]:
df_transformed.shape

In [ ]:
df_transformed.head()

In [ ]:
df_transformed.dropna(inplace=True)

In [ ]:
df_transformed.shape

In [ ]:
df_transformed.head()


In [ ]:
# First create the target
df_transformed['Target'] = df_transformed['Close'].shift(-1)

# Get the feature matrix (drop target columns)
X = df_transformed.drop(columns=['Target', 'Target_Change', 'Target_Binary'])

# Get the target
y = df_transformed['Target']

# Now drop rows where ANY of them have NaN
# This ensures X and y are aligned
df_clean = pd.concat([X, y], axis=1).dropna()

# Split back into X and y
X_clean = df_clean.drop(
    columns=['Target', 'Target_Change', 'Target_Binary'], errors='ignore')
y_clean = df_clean['Target']

print(f"X shape: {X_clean.shape}")
print(f"y shape: {y_clean.shape}")
print(f"Number of samples: {len(X_clean)}")

## Univariate Analysis

In [ ]:
# Box plot of Daily_Return - identify outliers
# Range of Price_Range

In [ ]:
# 1. Box Plot of Daily_Return to Identify Outliers
import seaborn as sns
import matplotlib.pyplot as plt

# Check if Daily_Return column exists
if 'Daily_Return' in df_transformed.columns:
    # Calculate statistics
    daily_returns = df_transformed['Daily_Return'].dropna()

    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Box plot
    bp = ax1.boxplot(daily_returns, vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue'),
                     medianprops=dict(color='red', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='red',
                                     markersize=5, alpha=0.6))

    ax1.set_title('Box Plot of Daily Returns', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Daily Return')
    ax1.grid(True, alpha=0.3, axis='y')

    # Add statistics text
    stats_text = f"""Statistics:
Mean: {daily_returns.mean():.4f}
Std: {daily_returns.std():.4f}
Min: {daily_returns.min():.4f}
25%: {daily_returns.quantile(0.25):.4f}
Median: {daily_returns.median():.4f}
75%: {daily_returns.quantile(0.75):.4f}
Max: {daily_returns.max():.4f}
IQR: {daily_returns.quantile(0.75) - daily_returns.quantile(0.25):.4f}
Outliers: {len(bp['fliers'][0].get_ydata())}"""

    ax1.text(1.02, 0.98, stats_text, transform=ax1.transAxes,
             verticalalignment='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    # Histogram with KDE
    ax2.hist(daily_returns, bins=50, alpha=0.7, color='skyblue',
             edgecolor='black', density=True)

    # Add KDE curve
    from scipy.stats import gaussian_kde
    if len(daily_returns) > 1:
        kde = gaussian_kde(daily_returns)
        x_range = np.linspace(daily_returns.min(), daily_returns.max(), 1000)
        ax2.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE')

    # Add vertical lines for statistics
    ax2.axvline(daily_returns.mean(), color='green', linestyle='--',
                linewidth=2, label=f'Mean: {daily_returns.mean():.4f}')
    ax2.axvline(daily_returns.median(), color='red', linestyle='--',
                linewidth=2, label=f'Median: {daily_returns.median():.4f}')

    ax2.set_title('Distribution of Daily Returns',
                  fontweight='bold', fontsize=12)
    ax2.set_xlabel('Daily Return')
    ax2.set_ylabel('Density')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Daily Returns Analysis with Outlier Detection',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Identify outliers using IQR method
    Q1 = daily_returns.quantile(0.25)
    Q3 = daily_returns.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = daily_returns[(daily_returns < lower_bound)
                             | (daily_returns > upper_bound)]

    print("="*60)
    print("DAILY RETURNS OUTLIER ANALYSIS")
    print("="*60)
    print(f"Total observations: {len(daily_returns)}")
    print(
        f"Outliers detected: {len(outliers)} ({len(outliers)/len(daily_returns)*100:.1f}%)")
    print(f"Lower bound: {lower_bound:.4f}")
    print(f"Upper bound: {upper_bound:.4f}")

    if len(outliers) > 0:
        print("\nTop 5 positive outliers:")
        print(outliers.nlargest(5).round(4))
        print("\nTop 5 negative outliers:")
        print(outliers.nsmallest(5).round(4))

else:
    print("Daily_Return column not found")

In [ ]:
# 2. Range of Price_Range Analysis
if 'Price_Range' in df_transformed.columns:
    # Calculate statistics
    price_range = df_transformed['Price_Range'].dropna()

    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Time series plot of Price_Range
    ax1.plot(df_transformed.index, price_range, color='purple',
             linewidth=1, alpha=0.7)
    ax1.axhline(y=price_range.mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'Mean: {price_range.mean():.3f}')

    # Add rolling average
    if len(price_range) > 20:
        rolling_mean = price_range.rolling(window=20).mean()
        ax1.plot(df_transformed.index, rolling_mean, color='green',
                 linewidth=2, label='20-Day MA')

    ax1.set_title('Price Range Over Time', fontweight='bold', fontsize=12)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Price Range (High - Low)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Histogram with statistics
    counts, bins, patches = ax2.hist(price_range, bins=50, alpha=0.7,
                                     color='orchid', edgecolor='black')

    # Add vertical lines for statistics
    ax2.axvline(price_range.mean(), color='red', linestyle='--',
                linewidth=2, label=f'Mean: {price_range.mean():.3f}')
    ax2.axvline(price_range.median(), color='green', linestyle='--',
                linewidth=2, label=f'Median: {price_range.median():.3f}')
    ax2.axvline(price_range.min(), color='blue', linestyle=':',
                linewidth=1, label=f'Min: {price_range.min():.3f}')
    ax2.axvline(price_range.max(), color='orange', linestyle=':',
                linewidth=1, label=f'Max: {price_range.max():.3f}')

    ax2.set_title('Distribution of Price Range',
                  fontweight='bold', fontsize=12)
    ax2.set_xlabel('Price Range (High - Low)')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Add statistics text
    range_stats = f"""Price Range Statistics:
Mean: {price_range.mean():.4f}
Std: {price_range.std():.4f}
Min: {price_range.min():.4f}
Max: {price_range.max():.4f}
Range: {price_range.max() - price_range.min():.4f}
25th %ile: {price_range.quantile(0.25):.4f}
75th %ile: {price_range.quantile(0.75):.4f}
IQR: {price_range.quantile(0.75) - price_range.quantile(0.25):.4f}
Skewness: {price_range.skew():.4f}
Kurtosis: {price_range.kurtosis():.4f}"""

    ax2.text(0.02, 0.98, range_stats, transform=ax2.transAxes,
             verticalalignment='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.8))

    plt.suptitle('Price Range Analysis: Daily Trading Range',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Additional analysis: Price Range vs Volatility
    print("="*60)
    print("PRICE RANGE ANALYSIS")
    print("="*60)
    print(f"Average daily price range: {price_range.mean():.4f}")
    print(
        f"Maximum daily range: {price_range.max():.4f} ({price_range.idxmax().date()})")
    print(
        f"Minimum daily range: {price_range.min():.4f} ({price_range.idxmin().date()})")
    print(f"Volatility of price range: {price_range.std():.4f}")

    # Check correlation with other volatility measures
    if 'Volatility_20' in df_transformed.columns:
        corr = price_range.corr(df_transformed['Volatility_20'].dropna())
        print(f"Correlation with 20-day volatility: {corr:.4f}")

    # Identify high and low volatility periods
    high_vol_days = price_range[price_range > price_range.quantile(0.75)]
    low_vol_days = price_range[price_range < price_range.quantile(0.25)]

    print(f"\nHigh volatility days (>75th percentile): {len(high_vol_days)}")
    print(f"Low volatility days (<25th percentile): {len(low_vol_days)}")

else:
    print("Price_Range column not found")
    # Check if we have High and Low columns to calculate it
    if all(col in df_transformed.columns for col in ['High', 'Low']):
        print("Calculating Price_Range from High and Low columns...")
        df_transformed['Price_Range'] = df_transformed['High'] - \
            df_transformed['Low']
        print("Price_Range created successfully")
        # Run the analysis again

## Bivariate Analysis

In [ ]:
# Correlation matrix: Heatmap of all features
# Close vs Day_of_Week (box plots by weekday)



In [ ]:
# 1. Correlation Matrix Heatmap of All Features
import seaborn as sns
import matplotlib.pyplot as plt

# Select only numeric columns for correlation
numeric_cols = df_transformed.select_dtypes(
    include=[np.number]).columns.tolist()
correlation_data = df_transformed[numeric_cols]

# Calculate correlation matrix
corr_matrix = correlation_data.corr()

# Create heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix,
            annot=False,  # Set to True to see values, but can be crowded
            cmap='coolwarm',
            center=0,
            square=True,
            cbar_kws={"shrink": 0.8},
            linewidths=0.5,
            linecolor='gray')

plt.title('Correlation Matrix Heatmap of All Features',
          fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Optional: Show top correlations
print("TOP 10 POSITIVE CORRELATIONS:")
top_pos_corr = (corr_matrix.unstack()
                .sort_values(ascending=False)
                .drop_duplicates()
                .head(11))  # Top 10 + self-correlation
for (feat1, feat2), value in top_pos_corr.items():
    if feat1 != feat2:  # Exclude self-correlations
        print(f"{feat1:20} ↔ {feat2:20}: {value:.3f}")

print("\nTOP 10 NEGATIVE CORRELATIONS:")
top_neg_corr = (corr_matrix.unstack()
                .sort_values(ascending=True)
                .head(10))
for (feat1, feat2), value in top_neg_corr.items():
    print(f"{feat1:20} ↔ {feat2:20}: {value:.3f}")

In [ ]:
# 2. Close vs Day_of_Week (Box Plots by Weekday)
import seaborn as sns
import matplotlib.pyplot as plt

# Check if Day_of_Week column exists
if 'Day_of_Week' in df_transformed.columns:
    # Map day numbers to names
    day_names = ['Monday', 'Tuesday', 'Wednesday',
                 'Thursday', 'Friday', 'Saturday', 'Sunday']

    # Create a copy with day names
    plot_data = df_transformed.copy()
    plot_data['Day_Name'] = plot_data['Day_of_Week'].map(
        lambda x: day_names[int(x)] if not pd.isna(x) else None)
    plot_data = plot_data.dropna(subset=['Day_Name', 'Close'])

    # Create box plot
    plt.figure(figsize=(12, 6))

    # Box plot
    sns.boxplot(x='Day_Name', y='Close', data=plot_data,
                order=day_names[:5] if plot_data['Day_of_Week'].max(
                ) < 5 else day_names,
                palette='Set2')

    # Add stripplot for individual points
    sns.stripplot(x='Day_Name', y='Close', data=plot_data,
                  order=day_names[:5] if plot_data['Day_of_Week'].max(
                  ) < 5 else day_names,
                  color='black', alpha=0.3, size=2, jitter=True)

    # Add mean line
    plt.axhline(y=plot_data['Close'].mean(), color='red', linestyle='--',
                linewidth=1, label=f"Overall Mean: {plot_data['Close'].mean():.3f}")

    plt.title('Close Price Distribution by Day of Week',
              fontsize=14, fontweight='bold')
    plt.xlabel('Day of Week')
    plt.ylabel('Close Price (Transformed)')
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    # Print statistics
    print("CLOSE PRICE STATISTICS BY DAY OF WEEK:")
    print("="*50)
    day_stats = plot_data.groupby('Day_Name')['Close'].agg([
        'mean', 'std', 'count'])
    print(day_stats.round(3))

    # Check for significant differences (ANOVA-like)
    from scipy import stats
    print("\n" + "="*50)
    print("STATISTICAL TEST FOR DAY-OF-WEEK EFFECT:")

    # Group data by day
    day_groups = [group['Close'].values for name,
                  group in plot_data.groupby('Day_Name')]

    if len(day_groups) >= 2:
        # Kruskal-Wallis test (non-parametric ANOVA)
        stat, p_value = stats.kruskal(*day_groups)
        print(f"Kruskal-Wallis H-test: H = {stat:.3f}, p = {p_value:.4f}")
        if p_value < 0.05:
            print("✅ Significant difference between days (p < 0.05)")
        else:
            print("❌ No significant difference between days")

else:
    print("Day_of_Week column not found in dataframe")
    print("Available columns:", list(df_transformed.columns)[:10], "...")

## Multivariate Analysis

In [ ]:
# 3D scatter plot with color by Target_Binary
# Group similar trading days using K-means on features




In [ ]:
# 3D Scatter Plot with Color by Target_Binary - Using available columns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Check if we have at least 3 suitable columns
available_cols = df_transformed.columns.tolist()

# Try to find 3 meaningful features (adjust based on your actual columns)
if 'Daily_Return' in available_cols and 'Volume' in available_cols and 'Price_Range' in available_cols:
    features_3d = ['Daily_Return', 'Volume', 'Price_Range']
elif 'Close' in available_cols and 'Volume' in available_cols and 'Volatility_20' in available_cols:
    features_3d = ['Close', 'Volume', 'Volatility_20']
elif 'MA_5' in available_cols and 'MA_20' in available_cols and 'Volume' in available_cols:
    features_3d = ['MA_5', 'MA_20', 'Volume']
else:
    # Use first 3 numeric columns
    numeric_cols = df_transformed.select_dtypes(
        include=[np.number]).columns.tolist()
    features_3d = numeric_cols[:3] if len(numeric_cols) >= 3 else numeric_cols

print(f"Using features for 3D plot: {features_3d}")

# Create 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Drop rows with NaN in selected features
plot_data = df_transformed[features_3d + ['Target_Binary']].dropna()

# Scatter plot colored by Target_Binary (0=red, 1=green)
scatter = ax.scatter(plot_data[features_3d[0]],
                     plot_data[features_3d[1]],
                     plot_data[features_3d[2]],
                     c=plot_data['Target_Binary'],
                     cmap='RdYlGn',
                     alpha=0.6,
                     s=20)

ax.set_xlabel(features_3d[0])
ax.set_ylabel(features_3d[1])
ax.set_zlabel(features_3d[2])
ax.set_title('3D Scatter: Market Behavior by Price Direction',
             fontweight='bold')

# Add colorbar
plt.colorbar(scatter, label='Price Direction (0=Down, 1=Up)')
plt.tight_layout()
plt.show()

In [ ]:
# SIMPLE K-means Clustering with what you definitely have
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Use columns that you definitely created earlier
simple_features = ['Close', 'Volume', 'Daily_Return']  # These should exist
df_cluster = df_transformed[simple_features].dropna()

print(f"Clustering data shape: {df_cluster.shape}")

if len(df_cluster) > 10:  # Need enough data
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_cluster)

    # Apply K-means
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    df_cluster['Cluster'] = clusters

    print("\nCluster centers:")
    print(kmeans.cluster_centers_)

    # Simple visualization
    plt.figure(figsize=(10, 6))
    for cluster in range(3):
        cluster_data = df_cluster[df_cluster['Cluster'] == cluster]
        plt.scatter(cluster_data['Close'], cluster_data['Volume'],
                    alpha=0.6, s=30, label=f'Cluster {cluster}')

    plt.xlabel('Close Price')
    plt.ylabel('Volume')
    plt.title('K-means Clustering Results (3 clusters)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data for clustering")

## Final Dataframe

In [ ]:
df_transformed.tail()

In [ ]:
print(df_transformed.columns.tolist())

# Machine Learning Models

<!-- ## ARIMA -->

ARIMA is a univariate time-series model, which means:

It models one time-dependent variable

It does not directly use multiple features like ML models (Random Forest, XGBoost, LSTM, etc.)

### Data Preparation

In [ ]:
df.tail()

### Original Close Price Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
%matplotlib inline
# Add grid for better readability
df['Close'].plot(
    figsize=(14, 6), title='Original Close Price (df)', color='blue', grid=True)

# Add axis labels
ax = df['Close'].plot(
    figsize=(14, 6), title='Original Close Price (df)', color='blue')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price')

# Show the plot (if not in Jupyter)
plt.show()

### Apply Transformation

In [ ]:
df_transformed = df.copy()

In [ ]:
df_transformed = np.log1p(df_transformed) # Log transformation
df_transformed['Close'].plot(
    figsize=(14, 6), title='Transformed Close Price (df_transformed)', color='green', grid=True)

### Spliting the data

In [ ]:
df_transformed.tail()

In [ ]:
train_size = len(df_transformed['Close']) - 30
df_train = df_transformed['Close'].iloc[:train_size]
df_test = df_transformed['Close'].iloc[train_size:]

In [ ]:
df_train.shape, df_test.shape

### PCF and PACF Plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Assuming df_transformed is your DataFrame
# Let's extract the Close price series
close_series = df_transformed['Close']

# Create subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot ACF (Autocorrelation Function)
plot_acf(close_series, lags=40,
         ax=axes[0], title='Autocorrelation Function (ACF) - Close Price')
axes[0].set_xlabel('Lags')
axes[0].set_ylabel('Autocorrelation')

# Plot PACF (Partial Autocorrelation Function)
plot_pacf(close_series, lags=40,
          ax=axes[1], title='Partial Autocorrelation Function (PACF) - Close Price')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('Partial Autocorrelation')

plt.tight_layout()
plt.show()

### ADF Test

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Augmented Dickey-Fuller test
result = adfuller(df_transformed['Close'])

print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.6f}")
print(f"Critical Values:")
for key, value in result[4].items():
    print(f"  {key}: {value:.4f}")

# Conclusion
if result[1] <= 0.05:
    print("\n✅ Series is STATIONARY (reject null hypothesis)")
else:
    print("\n❌ Series is NON-STATIONARY (fail to reject null hypothesis)")

### Transforming to Stationary: Differencing

In [ ]:
# First difference
df_diff = df_train.diff()

# Drop NaN from first row
df_diff = df_diff.dropna()

print(f"Original shape: {df_train.shape}")
print(f"Differenced shape: {df_diff.shape}")

# Quick check
print("\nFirst 5 values:")
print(df_diff.head())
df_diff.plot(
    figsize=(14, 6), title='First Difference of Transformed Close Price', color='orange', grid=True)

### Apply PCF and PACF plot and ADF test to the data

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Check stationarity of differenced series
adf_result = adfuller(df_diff)
p_value = adf_result[1]

# Create plots
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ACF Plot
plot_acf(df_diff, lags=40, ax=axes[0],
         title=f'ACF Plot - First Differenced Close Price (p={p_value:.6f})')

# PACF Plot
plot_pacf(df_diff, lags=40, ax=axes[1],
          title=f'PACF Plot - First Differenced Close Price (p={p_value:.6f})')

plt.tight_layout()
plt.show()

print(f"\nDifferenced Series ADF p-value: {p_value:.6f}")
print(f"Stationary: {'YES' if p_value <= 0.05 else 'NO'}")

### Determine ARIMA models parameters p, q

In [ ]:
# ARIMA 0, 1, 0 model (Random Walk with drift)
# The first difference removed all predictable patterns
# Daily price changes appear random (uncorrelated with past changes)



### Fit the ARIMA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Assuming you've already split the data
print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

# Apply ARIMA(0,1,0) based on our analysis
print("\nFitting ARIMA(0,1,0) model...")
model = ARIMA(df_train, order=(0, 1, 0))
model_fit = model.fit()

print("\n" + "="*50)
print("MODEL SUMMARY")
print("="*50)
print(model_fit.summary())

### Plotting Residual and Density Plot

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import shapiro, jarque_bera
from statsmodels.graphics.tsaplots import plot_acf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Get residuals from the fitted model
residuals = model_fit.resid

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals over time
axes[0, 0].plot(residuals.index, residuals, color='red', alpha=0.7)
axes[0, 0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[0, 0].set_title('Residuals Over Time', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].grid(True, alpha=0.3)

# 2. Histogram with KDE
axes[0, 1].hist(residuals, bins=30, density=True, alpha=0.7,
                color='skyblue', edgecolor='black')
sns.kdeplot(residuals, ax=axes[0, 1], color='darkred', linewidth=2)
axes[0, 1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0, 1].set_title('Residual Distribution with KDE',
                     fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Residual Value')
axes[0, 1].set_ylabel('Density')
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q Plot for normality check
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].get_lines()[0].set_marker('o')
axes[1, 0].get_lines()[0].set_markersize(4)
axes[1, 0].get_lines()[0].set_alpha(0.7)
axes[1, 0].get_lines()[1].set_color('red')
axes[1, 0].get_lines()[1].set_linewidth(2)
axes[1, 0].set_title('Q-Q Plot (Normality Check)',
                     fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. ACF of residuals (to check for remaining autocorrelation)
plot_acf(residuals, lags=40, ax=axes[1, 1], title='ACF of Residuals')
axes[1, 1].set_title('ACF of Residuals', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical tests for residuals
print("="*60)
print("RESIDUAL DIAGNOSTICS")
print("="*60)

# Normality test
shapiro_stat, shapiro_p = shapiro(residuals.dropna())
jb_stat, jb_p = jarque_bera(residuals.dropna())

print(f"\n1. Normality Tests:")
print(
    f"   Shapiro-Wilk test: statistic={shapiro_stat:.4f}, p-value={shapiro_p:.6f}")
print(f"   Jarque-Bera test: statistic={jb_stat:.4f}, p-value={jb_p:.6f}")
print(f"   Normal distribution? {'YES' if shapiro_p > 0.05 else 'NO'}")

# Zero mean test
mean_resid = residuals.mean()
std_resid = residuals.std()
print(f"\n2. Mean and Std Deviation:")
print(f"   Mean of residuals: {mean_resid:.6f} (should be ~0)")
print(f"   Std of residuals: {std_resid:.6f}")

# Ljung-Box test for autocorrelation
lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print(f"\n3. Ljung-Box Test (autocorrelation):")
print(f"   p-value at lag 10: {lb_test['lb_pvalue'].iloc[0]:.6f}")
print(
    f"   No autocorrelation? {'YES' if lb_test['lb_pvalue'].iloc[0] > 0.05 else 'NO'}")

print(f"\n4. Residual Summary:")
print(f"   Min: {residuals.min():.4f}")
print(f"   Max: {residuals.max():.4f}")
print(f"   Range: {residuals.max() - residuals.min():.4f}")

### Prediction and Forecasting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Make forecast for the test period
forecast_steps = len(df_test)  # Forecast same length as test set
forecast = model_fit.forecast(steps=forecast_steps)

# Create forecast series with proper index
forecast_index = pd.date_range(
    start=df_train.index[-1], periods=forecast_steps+1, freq='D')[1:]
forecast_series = pd.Series(forecast, index=forecast_index)

print("="*60)
print("FORECAST RESULTS")
print("="*60)
print(
    f"Forecast period: {forecast_series.index[0]} to {forecast_series.index[-1]}")
print(f"Number of forecast steps: {forecast_steps}")
print(f"\nFirst 5 forecast values:")
print(forecast_series.head())
print(f"\nLast 5 forecast values:")
print(forecast_series.tail())

# Calculate forecast accuracy metrics
mae = mean_absolute_error(df_test, forecast_series)
mse = mean_squared_error(df_test, forecast_series)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((df_test - forecast_series) / df_test)) * 100

print("\n" + "="*60)
print("FORECAST ACCURACY METRICS")
print("="*60)
print(f"Mean Absolute Error (MAE):      {mae:.4f}")
print(f"Mean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute % Error (MAPE):   {mape:.2f}%")

# Plot the forecast vs actual
plt.figure(figsize=(14, 8))

# Plot training data (last 100 observations for clarity)
train_plot_len = min(100, len(df_train))
plt.plot(df_train.index[-train_plot_len:], df_train[-train_plot_len:],
         label='Training Data', color='blue', linewidth=2)

# Plot test data
plt.plot(df_test.index, df_test, label='Actual Test Data',
         color='green', linewidth=2)

# Plot forecast
plt.plot(forecast_series.index, forecast_series, label='ARIMA Forecast',
         color='red', linestyle='--', linewidth=2)

plt.title(
    f'ARIMA(0,1,0) Forecast vs Actual\nMAPE: {mape:.2f}%', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price', fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Plot forecast error
forecast_error = df_test - forecast_series

plt.figure(figsize=(14, 5))
plt.bar(forecast_error.index, forecast_error, alpha=0.7, color='orange')
plt.axhline(y=0, color='black', linestyle='-', linewidth=1)
plt.title('Forecast Errors (Actual - Forecast)',
          fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Error')
plt.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Auto Fit The ARIMA Model

In [ ]:
import pmdarima as pm
# Super simple auto-ARIMA
auto_model = pm.auto_arima(df_train, seasonal=False, suppress_warnings=True)
forecast = auto_model.predict(n_periods=len(df_test))
print(f"Auto-ARIMA{auto_model.order} forecast complete! MAPE: {np.mean(np.abs((df_test - forecast)/df_test))*100:.2f}%")

### Evaluate

In [ ]:
from scipy.stats import jarque_bera, shapiro
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("="*70)
print("COMPARISON: MANUAL ARIMA vs AUTO-ARIMA")
print("="*70)

# 1. Manual ARIMA(0,1,0) model (from earlier)
print("\n🔵 MANUAL ARIMA(0,1,0) MODEL")
print("-"*40)
manual_model = ARIMA(df_train, order=(0, 1, 0)).fit()
manual_forecast = manual_model.forecast(steps=len(df_test))
manual_residuals = manual_model.resid

# Manual model metrics
manual_mae = mean_absolute_error(df_test, manual_forecast)
manual_rmse = np.sqrt(mean_squared_error(df_test, manual_forecast))
manual_mape = np.mean(np.abs((df_test - manual_forecast) / df_test)) * 100

print(f"Order: (0,1,0)")
print(f"AIC: {manual_model.aic:.2f}")
print(f"BIC: {manual_model.bic:.2f}")
print(f"MAE: {manual_mae:.4f}")
print(f"RMSE: {manual_rmse:.4f}")
print(f"MAPE: {manual_mape:.2f}%")

# 2. Auto-ARIMA model (from pmdarima)
print("\n🔴 AUTO-ARIMA MODEL")
print("-"*40)
auto_model = pm.auto_arima(df_train,
                           seasonal=False,
                           suppress_warnings=True,
                           trace=False)
auto_forecast = auto_model.predict(n_periods=len(df_test))
auto_residuals = auto_model.resid()

# Auto model metrics
auto_mae = mean_absolute_error(df_test, auto_forecast)
auto_rmse = np.sqrt(mean_squared_error(df_test, auto_forecast))
auto_mape = np.mean(np.abs((df_test - auto_forecast) / df_test)) * 100

print(f"Order: {auto_model.order}")
print(f"AIC: {auto_model.aic():.2f}")
print(f"BIC: {auto_model.bic():.2f}")
print(f"MAE: {auto_mae:.4f}")
print(f"RMSE: {auto_rmse:.4f}")
print(f"MAPE: {auto_mape:.2f}%")

# 3. Comparison summary
print("\n📊 COMPARISON SUMMARY")
print("-"*40)

# Calculate improvements
mape_improvement = ((manual_mape - auto_mape) / manual_mape) * 100
rmse_improvement = ((manual_rmse - auto_rmse) / manual_rmse) * 100
aic_difference = manual_model.aic - auto_model.aic()

print(
    f"MAPE Improvement: {mape_improvement:+.1f}% ({'Better' if auto_mape < manual_mape else 'Worse'})")
print(
    f"RMSE Improvement: {rmse_improvement:+.1f}% ({'Better' if auto_rmse < manual_rmse else 'Worse'})")
print(
    f"AIC Difference: {aic_difference:+.1f} ({'Better' if aic_difference > 0 else 'Worse'})")

if auto_mape < manual_mape:
    print(
        f"✅ Auto-ARIMA performs better by {manual_mape - auto_mape:.2f}% MAPE")
else:
    print(
        f"⚠️  Manual ARIMA performs better by {auto_mape - manual_mape:.2f}% MAPE")

# 4. Visual Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Forecast Comparison
axes[0, 0].plot(df_test.index, df_test, label='Actual',
                color='black', linewidth=2, alpha=0.8)
axes[0, 0].plot(df_test.index, manual_forecast, label='ARIMA(0,1,0)',
                color='blue', linestyle='--', linewidth=1.5)
axes[0, 0].plot(df_test.index, auto_forecast, label=f'Auto-ARIMA{auto_model.order}',
                color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('Forecast Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Close Price')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# Plot 2: Error Comparison
error_manual = df_test - manual_forecast
error_auto = df_test - auto_forecast

axes[0, 1].bar(df_test.index, error_manual, alpha=0.5, width=0.4,
               label=f'ARIMA(0,1,0) Error', color='blue')
axes[0, 1].bar(df_test.index + pd.Timedelta(days=0.4), error_auto, alpha=0.5, width=0.4,
               label=f'Auto-ARIMA Error', color='red')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[0, 1].set_title('Forecast Errors Comparison',
                     fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Error (Actual - Forecast)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].tick_params(axis='x', rotation=45)

# Plot 3: Residuals Comparison
axes[1, 0].hist(manual_residuals.dropna(), bins=30, alpha=0.5, label='ARIMA(0,1,0)',
                color='blue', density=True)
axes[1, 0].hist(auto_residuals, bins=30, alpha=0.5, label=f'Auto-ARIMA{auto_model.order}',
                color='red', density=True)
axes[1, 0].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1, 0].set_title('Residuals Distribution Comparison',
                     fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Residual Value')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Metrics Comparison
metrics = ['MAPE', 'RMSE', 'AIC']
manual_vals = [manual_mape, manual_rmse, manual_model.aic]
auto_vals = [auto_mape, auto_rmse, auto_model.aic()]

x = np.arange(len(metrics))
width = 0.35

axes[1, 1].bar(x - width/2, manual_vals, width,
               label='ARIMA(0,1,0)', color='blue', alpha=0.8)
axes[1, 1].bar(x + width/2, auto_vals, width,
               label=f'Auto-ARIMA{auto_model.order}', color='red', alpha=0.8)

axes[1, 1].set_title('Performance Metrics Comparison',
                     fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Metric')
axes[1, 1].set_ylabel('Value')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(metrics)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (m, a) in enumerate(zip(manual_vals, auto_vals)):
    axes[1, 1].text(i - width/2, m + max(max(manual_vals), max(auto_vals))*0.01,
                    f'{m:.1f}' if i < 2 else f'{m:.0f}',
                    ha='center', va='bottom', fontsize=9)
    axes[1, 1].text(i + width/2, a + max(max(manual_vals), max(auto_vals))*0.01,
                    f'{a:.1f}' if i < 2 else f'{a:.0f}',
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 5. Detailed Residual Analysis
print("\n" + "="*70)
print("RESIDUAL ANALYSIS COMPARISON")
print("="*70)


def analyze_residuals(residuals, name):
    print(f"\n📈 {name} Residuals:")
    print(f"  Mean: {residuals.mean():.6f}")
    print(f"  Std: {residuals.std():.6f}")
    print(f"  Skewness: {residuals.skew():.4f}")
    print(f"  Kurtosis: {residuals.kurtosis():.4f}")

    # Normality tests
    jb_stat, jb_p = jarque_bera(residuals.dropna() if hasattr(
        residuals, 'dropna') else residuals)
    print(
        f"  Jarque-Bera p-value: {jb_p:.6f} ({'Normal' if jb_p > 0.05 else 'Not Normal'})")

    # Ljung-Box test for autocorrelation
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
    print(f"  Ljung-Box p-value (lag 10): {lb_test['lb_pvalue'].iloc[0]:.6f} "
          f"({'White Noise' if lb_test['lb_pvalue'].iloc[0] > 0.05 else 'Autocorrelated'})")


analyze_residuals(manual_residuals, "ARIMA(0,1,0)")
analyze_residuals(pd.Series(auto_residuals), f"Auto-ARIMA{auto_model.order}")

# 6. Final Recommendation
print("\n" + "="*70)
print("FINAL RECOMMENDATION")
print("="*70)

if auto_mape < manual_mape and auto_model.aic() < manual_model.aic:
    print(f"✅ RECOMMEND: Auto-ARIMA{auto_model.order}")
    print(f"   Reasons:")
    print(f"   1. Lower MAPE ({auto_mape:.2f}% vs {manual_mape:.2f}%)")
    print(
        f"   2. Lower AIC ({auto_model.aic():.0f} vs {manual_model.aic:.0f})")
    print(f"   3. Better fit based on information criteria")
elif manual_mape < auto_mape and manual_model.aic < auto_model.aic():
    print(f"✅ RECOMMEND: Manual ARIMA(0,1,0)")
    print(f"   Reasons:")
    print(f"   1. Lower MAPE ({manual_mape:.2f}% vs {auto_mape:.2f}%)")
    print(
        f"   2. Lower AIC ({manual_model.aic:.0f} vs {auto_model.aic():.0f})")
    print(f"   3. Simpler model with comparable performance")
else:
    print("🤔 CONSIDER BOTH MODELS")
    print(f"   Trade-off between simplicity and accuracy")
    print(f"   ARIMA(0,1,0): Simpler, MAPE={manual_mape:.2f}%")
    print(
        f"   Auto-ARIMA{auto_model.order}: More complex, MAPE={auto_mape:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create figure with subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Chart 1: ARIMA(0,1,0) Model
axes[0].plot(df_train.index[-50:], df_train[-50:],
             label='Training Data', color='blue', linewidth=2, alpha=0.8)
axes[0].plot(df_test.index, df_test,
             label='Actual Test Data', color='green', linewidth=2, alpha=0.8)
axes[0].plot(df_test.index, manual_forecast,
             label='ARIMA Forecast', color='red', linestyle='--', linewidth=2)

axes[0].set_title(f'ARIMA(0,1,0) Forecast vs Actual\nMAPE: {manual_mape:.2f}%',
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Close Price', fontsize=12)
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Auto-ARIMA Model
axes[1].plot(df_train.index[-50:], df_train[-50:],
             label='Training Data', color='blue', linewidth=2, alpha=0.8)
axes[1].plot(df_test.index, df_test,
             label='Actual Test Data', color='green', linewidth=2, alpha=0.8)
axes[1].plot(df_test.index, auto_forecast,
             label=f'Auto-ARIMA{auto_model.order} Forecast',
             color='red', linestyle='--', linewidth=2)

axes[1].set_title(f'Auto-ARIMA{auto_model.order} Forecast vs Actual\nMAPE: {auto_mape:.2f}%',
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Close Price', fontsize=12)
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## LSTM

### Read CSV and Setting Index Column

In [ ]:
df = read_csv('../data/BTC-USD/BTC-USD_from_2010-01-01_till_2026-01-03.csv')
df.tail()

In [ ]:
# Set Date as index for time series analysis

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.shape

In [ ]:
df.tail()

In [ ]:
df = df[['Close']]
df.tail()

### Visualization

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.plot(df.index, df['Close'], color='blue')
plt.show()

### Preprocessing

In [ ]:
import numpy as np
import pandas as pd


def df_to_windowed_df(dataframe, first_date_str, last_date_str, window_size=5):
    """Convert DataFrame to windowed DataFrame for time series analysis."""

    first_date = pd.to_datetime(first_date_str)
    last_date = pd.to_datetime(last_date_str)

    # Filter dataframe by date range
    df_filtered = dataframe.loc[
        (dataframe.index >= first_date) & (dataframe.index <= last_date)
    ].copy()

    # Create lagged features (previous date values)
    for i in range(1, window_size + 1):
        df_filtered[f'Close_lag_{i}'] = df_filtered['Close'].shift(i)

    # Target is the current date's Close value
    df_filtered['Target'] = df_filtered['Close']

    # Drop rows with NaN values created by shifting
    df_windowed = df_filtered.dropna()

    # Reorder columns: Close_lag_5, Close_lag_4, ..., Close_lag_1, Target
    lag_cols = [f'Close_lag_{i}' for i in range(window_size, 0, -1)]
    df_windowed = df_windowed[lag_cols + ['Target']]

    return df_windowed


df_windowed = df_to_windowed_df(df, '2024-12-31', '2025-12-31', window_size=5)
df_windowed.tail()

In [ ]:
df_windowed.shape

In [ ]:
def windowed_df_to_date_X_y(windowed_dataframe):
    """
    Convert windowed DataFrame into dates, X, y arrays.
    Assumes:
    - Index is Date
    - Columns: Close_lag_n ... Close_lag_1, Target
    """

    # Dates come from the index
    dates = windowed_dataframe.index.to_numpy()

    # Convert dataframe to numpy
    df_as_np = windowed_dataframe.to_numpy()

    # X = all lag columns (everything except last column)
    X = df_as_np[:, :-1]

    print(f"X shape before reshape: {X.shape}")

    # Reshape X to (samples, timesteps, features)
    X = X.reshape((len(dates), X.shape[1], 1))

    print(f"X shape after reshape: {X.shape}")
    # y = Target column
    y = df_as_np[:, -1]

    return dates, X.astype(np.float32), y.astype(np.float32)

In [ ]:
dates, X, y = windowed_df_to_date_X_y(df_windowed)
dates.shape, X.shape, y.shape

### Spliting Dataset into train and test

In [ ]:
q_80 = int(len(dates) * .8)
q_90 = int(len(dates) * .9)

dates_train, X_train, y_train = dates[:q_80], X[:q_80], y[:q_80]
dates_val, X_val, y_val = dates[q_80:q_90], X[q_80:q_90], y[q_80:q_90]
dates_test, X_test, y_test = dates[q_90:], X[q_90:], y[q_90:]


plt.plot(dates_train, y_train, label='Train')
plt.plot(dates_val, y_val, label='Validation')
plt.plot(dates_test, y_test, label='Test')
plt.legend()
plt.show()

### Train The Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.optimizers import Adam
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
print("="*60)
print("DATA INSPECTION")
print("="*60)

print("Data shapes before processing:")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

print(f"\nX_train sample (first 5):")
print(X_train[:5])
print(f"\ny_train sample (first 5):")
print(y_train[:5])

print(f"\nDates ranges:")
print(f"Training: {dates_train[0]} to {dates_train[-1]}")
print(f"Validation: {dates_val[0]} to {dates_val[-1]}")
print(f"Test: {dates_test[0]} to {dates_test[-1]}")

In [ ]:
print("\n" + "="*60)
print("DATA SCALING")
print("="*60)

scaler_X = StandardScaler()
scaler_y = StandardScaler()


def scale_data(X_train, X_val, X_test, y_train, y_val, y_test):
    """Scale features and target variables."""

    # Scale X features
    if len(X_train.shape) == 3:
        X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
        X_val_reshaped = X_val.reshape(-1, X_val.shape[-1])
        X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])

        X_train_scaled = scaler_X.fit_transform(
            X_train_reshaped).reshape(X_train.shape)
        X_val_scaled = scaler_X.transform(X_val_reshaped).reshape(X_val.shape)
        X_test_scaled = scaler_X.transform(
            X_test_reshaped).reshape(X_test.shape)
    else:
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_val_scaled = scaler_X.transform(X_val)
        X_test_scaled = scaler_X.transform(X_test)

    # Scale y target
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

    return X_train_scaled, X_val_scaled, X_test_scaled, y_train_scaled, y_val_scaled, y_test_scaled


# Apply scaling
X_train_scaled, X_val_scaled, X_test_scaled, y_train_scaled, y_val_scaled, y_test_scaled = scale_data(
    X_train, X_val, X_test, y_train, y_val, y_test
)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(
    f"X_train_scaled mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(
    f"y_train_scaled mean: {y_train_scaled.mean():.4f}, std: {y_train_scaled.std():.4f}")

In [ ]:
print("\n" + "="*60)
print("BUILDING LSTM MODEL")
print("="*60)


def build_lstm_model(input_shape=(3, 1)):
    """Build and return an LSTM model."""

    model = Sequential([
        layers.Input(input_shape),
        layers.LSTM(128, return_sequences=True,
                    kernel_initializer='glorot_uniform'),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=True,
                    kernel_initializer='glorot_uniform'),
        layers.Dropout(0.2),
        layers.LSTM(32, kernel_initializer='glorot_uniform'),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(8, activation='relu'),
        layers.Dense(1)
    ])

    optimizer = Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-07
    )

    model.compile(
        loss='mse',
        optimizer=optimizer,
        metrics=['mean_absolute_error', 'mean_squared_error']
    )

    return model


# Build model
model = build_lstm_model(input_shape=(
    X_train_scaled.shape[1], X_train_scaled.shape[2]))
print("Model Summary:")
model.summary()

In [ ]:
print("\n" + "="*60)
print("SETTING UP TRAINING CALLBACKS")
print("="*60)


def get_training_callbacks():
    """Create and return training callbacks."""

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=0.00001,
        verbose=1
    )

    return [early_stopping, reduce_lr]


callbacks = get_training_callbacks()
print("Callbacks created: EarlyStopping, ReduceLROnPlateau")

In [ ]:
print("\n" + "="*60)
print("TRAINING MODEL")
print("="*60)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining completed after {len(history.history['loss'])} epochs")
print(f"Final training loss: {history.history['loss'][-1]:.4f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.4f}")

### Make Predictions

In [ ]:
print("\n" + "="*60)
print("MAKING PREDICTIONS")
print("="*60)


def make_predictions(model, X_train_scaled, X_val_scaled, X_test_scaled):
    """Make predictions on all datasets."""

    train_predictions_scaled = model.predict(
        X_train_scaled, verbose=0).flatten()
    val_predictions_scaled = model.predict(X_val_scaled, verbose=0).flatten()
    test_predictions_scaled = model.predict(X_test_scaled, verbose=0).flatten()

    # Inverse transform to original scale
    train_predictions = scaler_y.inverse_transform(
        train_predictions_scaled.reshape(-1, 1)).flatten()
    val_predictions = scaler_y.inverse_transform(
        val_predictions_scaled.reshape(-1, 1)).flatten()
    test_predictions = scaler_y.inverse_transform(
        test_predictions_scaled.reshape(-1, 1)).flatten()

    return train_predictions, val_predictions, test_predictions


train_predictions, val_predictions, test_predictions = make_predictions(
    model, X_train_scaled, X_val_scaled, X_test_scaled
)

print(f"Predictions made on {len(train_predictions)} training samples")
print(f"Predictions made on {len(val_predictions)} validation samples")
print(f"Predictions made on {len(test_predictions)} test samples")

### Model Visualization

In [ ]:
print("\n" + "="*60)
print("PLOTTING RESULTS")
print("="*60)


def plot_predictions(dates_train, y_train, train_predictions,
                     dates_val, y_val, val_predictions,
                     dates_test, y_test, test_predictions):
    """Plot predictions vs actual values."""

    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    # Plot training data
    axes[0].plot(dates_train, y_train, 'b-', label='Training Observations',
                 linewidth=2, alpha=0.7)
    axes[0].plot(dates_train, train_predictions, 'r--',
                 label='Training Predictions', linewidth=2)
    axes[0].set_title('Training Set: Predictions vs Actual', fontsize=14)
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Value')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    # Plot validation and test
    axes[1].plot(dates_val, y_val, 'g-', label='Validation Observations',
                 linewidth=2, alpha=0.7)
    axes[1].plot(dates_val, val_predictions, 'm--',
                 label='Validation Predictions', linewidth=2)
    axes[1].plot(dates_test, y_test, 'c-', label='Test Observations',
                 linewidth=2, alpha=0.7)
    axes[1].plot(dates_test, test_predictions, 'y--',
                 label='Test Predictions', linewidth=2)
    axes[1].set_title(
        'Validation & Test Sets: Predictions vs Actual', fontsize=14)
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Value')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    return fig


# Create predictions plot
prediction_plot = plot_predictions(
    dates_train, y_train, train_predictions,
    dates_val, y_val, val_predictions,
    dates_test, y_test, test_predictions
)

### Model Evaluation

In [ ]:
print("\n" + "="*60)
print("EVALUATION METRICS")
print("="*60)


def calculate_metrics(y_true, y_pred, dataset_name):
    """Calculate and display evaluation metrics."""

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{dataset_name} Set:")
    print(f"  MSE: {mse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  Predictions range: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
    print(f"  Actual range: [{y_true.min():.4f}, {y_true.max():.4f}]")

    return {'mse': mse, 'mae': mae, 'rmse': rmse, 'r2': r2}


# Calculate metrics for all datasets
metrics = {}
metrics['train'] = calculate_metrics(y_train, train_predictions, "Training")
metrics['val'] = calculate_metrics(y_val, val_predictions, "Validation")
metrics['test'] = calculate_metrics(y_test, test_predictions, "Test")

# Print summary
print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"{'Dataset':<15} {'MSE':<10} {'MAE':<10} {'RMSE':<10} {'R²':<10}")
print("-"*60)
for dataset in ['train', 'val', 'test']:
    m = metrics[dataset]
    print(
        f"{dataset.capitalize():<15} {m['mse']:<10.4f} {m['mae']:<10.4f} {m['rmse']:<10.4f} {m['r2']:<10.4f}")

## PROPHET

### Reading CSV File

In [ ]:
df = read_csv('../data/BTC-USD/BTC-USD_from_2010-01-01_till_2026-01-03.csv')
df.tail()

### Visualization

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.plot(df.index, df['Close'], color='blue')
plt.show()

###  Imports

In [ ]:
from prophet import Prophet

### Renaming the columns

In [ ]:
df_prophet = df.copy()

df_prophet["ds"] = pd.to_datetime(df_prophet["Date"])
df_prophet["y"] = df_prophet["Close"]

df_prophet = df_prophet[["ds", "y"]].sort_values("ds").reset_index(drop=True)

In [ ]:
df_prophet.tail()

### Training the Model

In [ ]:
prophet = Prophet(daily_seasonality=True, yearly_seasonality=True)
prophet.fit(df_prophet)

### Future predictions

In [ ]:
future = model.make_future_dataframe(periods=30)  # Forecast 30 days into future
forecast = model.predict(future)

### Plot

In [ ]:
# 5. Plot (matplotlib only - no Plotly)
fig = model.plot(forecast)
plt.title('Prophet Forecast')
plt.tight_layout()
plt.show()

### Evaluate on Test Data

## RANDOM FOREST

In [ ]:
df = read_csv('../data/BTC-USD/BTC-USD_from_2010-01-01_till_2026-01-03.csv')
df.tail()

In [ ]:
df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

### Defining Features and Target

In [ ]:
# Create lag features
def create_lags(df, column='Close', lags=5):
    df = df.copy()
    for lag in range(1, lags+1):
        df[f'lag_{lag}'] = df[column].shift(lag)
    df['target'] = df[column].shift(-1)  # Predict next day
    return df.dropna()


# Apply
df_features = create_lags(df, 'Close', lags=5)
X = df_features.drop(['Date', 'Close', 'target'], axis=1, errors='ignore')
y = df_features['target']

### Test Train Split

In [ ]:
# Time series split (80/20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df_features['Date'].iloc[split_idx:]

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

### Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained ✓")

### Predict and Evaluate

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: ${rmse:,.2f}")
print(f"Avg Price: ${y_test.mean():,.2f}")
print(f"RMSE % of Avg: {rmse/y_test.mean()*100:.1f}%")

### Plots

In [ ]:
# Simple plot
plt.figure(figsize=(12, 5))
plt.plot(dates_test, y_test, 'b-', label='Actual', linewidth=2)
plt.plot(dates_test, y_pred, 'r--', label='Predicted', linewidth=2)
plt.title('Random Forest Predictions vs Actual')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Forecast Predictions

In [ ]:
# Predict next 7 days
def predict_next_days(model, last_features, days=7):
    predictions = []
    current = last_features.copy()

    for _ in range(days):
        pred = model.predict(current.reshape(1, -1))[0]
        predictions.append(pred)
        # Update lags (shift by 1)
        current = np.roll(current, 1)
        current[0] = pred

    return predictions


# Get last row and predict
last_row = X.iloc[-1].values
next_prices = predict_next_days(model, last_row, days=7)
print(f"Next 7 day predictions: {[f'${p:,.0f}' for p in next_prices]}")





In [ ]:
# Update to predict 365 days (1 year)
next_prices_1yr = predict_next_days(model, last_row, days=365)

# Create future dates for 1 year
future_dates_1yr = pd.date_range(start=df['Date'].iloc[-1] + pd.Timedelta(days=1),
                                 periods=365, freq='D')

# 1. Plot COMPLETE historical + 1-year forecast
plt.figure(figsize=(16, 8))

# Plot ALL historical data
plt.plot(df['Date'], df['Close'], 'b-',
         label='Historical Data', linewidth=1.5, alpha=0.8)

# Plot 1-year forecast (line only, no markers for cleaner look)
plt.plot(future_dates_1yr, next_prices_1yr, 'r-',
         label='1-Year Forecast', linewidth=2)

# Add shaded forecast area
plt.axvspan(df['Date'].iloc[-1], future_dates_1yr[-1],
            alpha=0.15, color='red', label='Forecast Period (1 Year)')

# Add vertical line at forecast start
plt.axvline(x=df['Date'].iloc[-1], color='black', linestyle='--',
            label='Today', alpha=0.7, linewidth=1)

# Formatting
plt.title('COMPLETE PRICE HISTORY WITH 1-YEAR FORECAST',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

# Add forecast annotation
forecast_mid_date = future_dates_1yr[180]  # Middle of forecast
forecast_mid_price = next_prices_1yr[180]
plt.annotate('1-YEAR FORECAST',
             xy=(forecast_mid_date, forecast_mid_price),
             xytext=(20, 30),
             textcoords='offset points',
             fontsize=14,
             fontweight='bold',
             color='red',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.9))

plt.tight_layout()
plt.show()

# 2. ZOOM: Last 2 years + 1-year forecast
plt.figure(figsize=(16, 8))

# Get last 2 years of historical data
last_2y_dates = df['Date'] >= df['Date'].iloc[-1] - pd.Timedelta(days=730)
last_2y_data = df[last_2y_dates]

plt.plot(last_2y_data['Date'], last_2y_data['Close'], 'b-',
         label='Last 2 Years', linewidth=2.5)

# Plot 1-year forecast
plt.plot(future_dates_1yr, next_prices_1yr, 'r-',
         label='1-Year Forecast', linewidth=3)

# Connect historical to forecast
plt.plot([last_2y_data['Date'].iloc[-1], future_dates_1yr[0]],
         [last_2y_data['Close'].iloc[-1], next_prices_1yr[0]],
         'r--', linewidth=2, alpha=0.8)

# Highlight quarterly forecast points
quarter_dates = [future_dates_1yr[0], future_dates_1yr[90],
                 future_dates_1yr[180], future_dates_1yr[270], future_dates_1yr[-1]]
quarter_prices = [next_prices_1yr[0], next_prices_1yr[90],
                  next_prices_1yr[180], next_prices_1yr[270], next_prices_1yr[-1]]

plt.scatter(quarter_dates, quarter_prices,
            color='red', s=100, zorder=5,
            label='Quarterly Forecast Points')

# Add labels for quarterly points
labels = ['Today', '3 Months', '6 Months', '9 Months', '1 Year']
for date, price, label in zip(quarter_dates, quarter_prices, labels):
    plt.annotate(f'{label}\n${price:,.0f}',
                 xy=(date, price),
                 xytext=(0, 20),
                 textcoords='offset points',
                 ha='center',
                 fontsize=10,
                 fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='yellow', alpha=0.9))

plt.title('ZOOM: Last 2 Years + 1-Year Detailed Forecast',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3. FORECAST SUMMARY TABLE
print("\n" + "="*70)
print("1-YEAR FORECAST SUMMARY")
print("="*70)
current_price = df['Close'].iloc[-1]
print(f"Current Price (Today): ${current_price:,.2f}")
print(
    f"Historical Period: {df['Date'].iloc[0].strftime('%Y-%m-%d')} to {df['Date'].iloc[-1].strftime('%Y-%m-%d')}")
print(
    f"Forecast Period: {future_dates_1yr[0].strftime('%Y-%m-%d')} to {future_dates_1yr[-1].strftime('%Y-%m-%d')}")
print("="*70)

# Key milestone dates
milestones = [30, 90, 180, 270, 365]  # days
print(f"\n{'Milestone':<12} {'Date':<15} {'Predicted Price':<18} {'Change $':<15} {'Change %':<10}")
print("-"*70)

for days in milestones:
    date = future_dates_1yr[days-1]
    price = next_prices_1yr[days-1]
    change = price - current_price
    change_pct = (change / current_price) * 100

    if days == 30:
        label = "1 Month"
    elif days == 90:
        label = "3 Months"
    elif days == 180:
        label = "6 Months"
    elif days == 270:
        label = "9 Months"
    else:
        label = "1 Year"

    print(f"{label:<12} {date.strftime('%Y-%m-%d'):<15} ${price:<17,.0f} {change:>+14,.0f} {change_pct:>+9.1f}%")

print("="*70)

# 4. FORECAST TREND ANALYSIS
print("\n📊 FORECAST TREND ANALYSIS")
print("-"*50)

# Calculate min, max, and end values
forecast_min = min(next_prices_1yr)
forecast_max = max(next_prices_1yr)
forecast_end = next_prices_1yr[-1]

min_day = next_prices_1yr.index(forecast_min) + 1
max_day = next_prices_1yr.index(forecast_max) + 1

print(
    f"Peak Forecast:   Day {max_day} (${forecast_max:,.0f}) → {((forecast_max-current_price)/current_price*100):+.1f}%")
print(
    f"Lowest Forecast: Day {min_day} (${forecast_min:,.0f}) → {((forecast_min-current_price)/current_price*100):+.1f}%")
print(
    f"Year-End Price:  ${forecast_end:,.0f} → {((forecast_end-current_price)/current_price*100):+.1f}%")

# Calculate average forecast
forecast_avg = sum(next_prices_1yr) / len(next_prices_1yr)
print(
    f"Average Forecast: ${forecast_avg:,.0f} → {((forecast_avg-current_price)/current_price*100):+.1f}%")

# 5. MONTHLY FORECAST CHART
plt.figure(figsize=(14, 6))

# Extract monthly forecasts (approximate)
monthly_dates = []
monthly_prices = []
for i in range(0, 365, 30):  # Every ~30 days
    monthly_dates.append(future_dates_1yr[i])
    monthly_prices.append(next_prices_1yr[i])

# Add current price as month 0
monthly_dates = [df['Date'].iloc[-1]] + monthly_dates
monthly_prices = [current_price] + monthly_prices

plt.plot(monthly_dates, monthly_prices, 'go-',
         linewidth=2, markersize=8, label='Monthly Forecast')

# Fill between current and forecast
plt.fill_between(monthly_dates, current_price, monthly_prices,
                 alpha=0.1, color='green')

# Add value labels
for i, (date, price) in enumerate(zip(monthly_dates, monthly_prices)):
    if i == 0:
        label = f'Now\n${price:,.0f}'
    else:
        label = f'M{i}\n${price:,.0f}'

    plt.annotate(label,
                 xy=(date, price),
                 xytext=(0, 15),
                 textcoords='offset points',
                 ha='center',
                 fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.title('MONTHLY FORECAST PROGRESSION (Next 12 Months)',
          fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()